# Centinela QPU: sondeo previo de ~7 s antes de un trabajo caro

Qué resuelve: la calibración publicada de IBM se actualiza ~1 vez al día, pero en tus corridas v4
el qubit q142 de `ibm_fez` alternó en minutos entre **T2\* ≈ 155 µs** y **≈ 40 µs**, con un salto de
frecuencia de hasta 6 kHz (firma de un defecto TLS fluctuante). Enviar un trabajo largo en el estado
degradado desperdicia tiempo de QPU.

Qué hace, en **un job** y con **K candidatos en paralelo** (mismo costo que uno):
| Medición | Circuitos | Para qué |
|---|---|---|
| Ramsey en X e Y (tramo denso ≤ 10 µs + 2 largos) | 20 | detuning sin alias y T2\* |
| CPMG en 2 tiempos | 4 | cuánto recupera el DD **ahora** |
| T1 en 2 tiempos | 2 | separa desfase puro: Γφ = 1/T2\* − 1/(2T1) |
| Calibración de lectura | 2 | mitigación por qubit |

Salida: estado (`estable` / `degradado`) contra la huella histórica de cada qubit, ranking en vivo,
ganancia esperada del DD y la **corrección de fase** (Z virtual, costo cero) para tu circuito.
Cada sondeo se guarda en una bitácora JSONL: con el tiempo es la estadística de estados por qubit.


## 1. Instalación

In [ ]:
%pip install -q "qiskit>=1.3" "qiskit-ibm-runtime>=0.30" qiskit-aer matplotlib scipy
print("OK")

## 2. Paquete `ddqpu` embebido

In [ ]:
import pathlib, sys
_FILES = {}
_FILES['__init__.py'] = '"""ddqpu: validacion de Dynamical Decoupling en QPU IBM con control de relevos."""\n__version__ = "5.0.0"\nfrom .circuits import *  # noqa\nfrom .analysis import *  # noqa\nfrom .backend_utils import *  # noqa\nfrom .experiment import *  # noqa\nfrom . import sentinel  # noqa\n'
_FILES['circuits.py'] = '"""Construccion de circuitos DD con espectadores (vecinos + control) y tiempos alineados a dt.\n\nConvencion de bits: bit 0 = objetivo, bits 1..k = vecinos, ultimo bit = control lejano.\nTodos los qubits del layout se miden en todos los circuitos, asi cada circuito aporta datos\nde "relevo" (poblacion en espectadores) sin coste extra de circuitos.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom typing import List, Sequence\n\nimport numpy as np\nfrom qiskit import QuantumCircuit\n\nPROTOCOLS = ("free", "cpmg", "xy4")\nBASES = ("X", "Y")\n\n\n@dataclass\nclass Timing:\n    """Restricciones de tiempo del backend (todo en unidades de dt)."""\n    dt: float                 # segundos por muestra\n    granularity: int = 16     # multiplo obligatorio de duracion de delay\n    min_length: int = 64\n    x_duration_dt: int = 0    # duracion de un pulso X/Y (para t_efectivo)\n\n    def us_to_dt(self, t_us: float) -> int:\n        """Redondea t_us al multiplo de granularidad mas cercano (evita los UserWarning de redondeo\n        y garantiza que libre/CPMG/XY4 tengan exactamente el mismo tiempo de espera)."""\n        n = int(round(t_us * 1e-6 / self.dt))\n        g = max(int(self.granularity), 1)\n        return int(round(n / g) * g)\n\n    def dt_to_us(self, n: int) -> float:\n        return n * self.dt * 1e6\n\n\ndef timing_from_backend(backend) -> Timing:\n    target = backend.target\n    dt = float(target.dt or backend.dt)\n    gran, minlen = 16, 64\n    try:\n        tc = target.timing_constraints()\n        gran, minlen = int(tc.granularity), int(tc.min_length)\n    except Exception:\n        pass\n    xdur = 0\n    try:\n        q0 = next(iter(target["x"].keys()))\n        d = target["x"][q0].duration\n        if d:\n            xdur = int(round(d / dt))\n    except Exception:\n        pass\n    return Timing(dt=dt, granularity=max(gran, 1), min_length=max(minlen, 1), x_duration_dt=xdur)\n\n\n@dataclass\nclass CircuitSpec:\n    """Metadatos de un circuito del lote (se serializan en el manifiesto del job)."""\n    kind: str                 # main | cal0 | cal1 | idle | exc1 | zz | nsweep\n    protocol: str = "none"    # free | cpmg | xy4 | none\n    basis: str = "Z"          # X | Y | Z\n    t_us: float = 0.0         # tiempo de espera nominal alineado a dt\n    t_eff_us: float = 0.0     # tiempo total incluyendo pulsos DD\n    n_pulses: int = 0\n    spectator_state: str = "0"  # estado preparado en los vecinos\n    shots: int = 0\n    extra: dict = field(default_factory=dict)\n\n    def to_dict(self):\n        return asdict(self)\n\n\ndef _pulse_sequence(protocol: str, n_cpmg: int, n_xy4_cycles: int) -> List[str]:\n    if protocol == "cpmg":\n        return ["x"] * n_cpmg\n    if protocol == "xy4":\n        return ["x", "y", "x", "y"] * max(n_xy4_cycles, 1)\n    return []\n\n\ndef _dd_block(qc: QuantumCircuit, q: int, total_dt: int, pulses: Sequence[str], timing: Timing):\n    """tau/2 - P - tau - P - ... - P - tau/2, con todos los delays en dt y multiplos de granularidad.\n    El residuo de redondeo se absorbe en el ultimo tramo para conservar el tiempo total exacto."""\n    if total_dt <= 0:\n        return\n    if not pulses:\n        qc.delay(total_dt, q, unit="dt")\n        return\n    n = len(pulses)\n    g = timing.granularity\n    half = int((total_dt / (2 * n)) // g * g)\n    full = 2 * half\n    used = half + full * (n - 1) + half\n    last = half + (total_dt - used)\n    if half > 0:\n        qc.delay(half, q, unit="dt")\n    for i, p in enumerate(pulses):\n        getattr(qc, p)(q)\n        seg = full if i < n - 1 else last\n        if seg > 0:\n            qc.delay(seg, q, unit="dt")\n\n\ndef build_coherence_circuit(\n    protocol: str,\n    t_us: float,\n    basis: str,\n    n_spectators: int,\n    timing: Timing,\n    n_cpmg: int = 6,\n    n_xy4_cycles: int = 2,\n    spectator_state: str = "0",\n    n_neighbors: int = 2,\n):\n    """Objetivo en |+>, espera con/sin DD, lectura en base X o Y. Espectadores inactivos."""\n    n = 1 + n_spectators\n    t_dt = timing.us_to_dt(t_us)\n    pulses = _pulse_sequence(protocol, n_cpmg, n_xy4_cycles) if t_dt > 0 else []\n    qc = QuantumCircuit(n, n, name=f"{protocol}_{basis}_{t_dt}dt_s{spectator_state}")\n    if spectator_state == "1":\n        for k in range(1, 1 + n_neighbors):\n            qc.x(k)\n    qc.h(0)\n    qc.barrier()\n    _dd_block(qc, 0, t_dt, pulses, timing)\n    qc.barrier()\n    if basis == "Y":\n        qc.sdg(0)\n    qc.h(0)\n    qc.measure(range(n), range(n))\n    t_eff = timing.dt_to_us(t_dt + len(pulses) * timing.x_duration_dt)\n    spec = CircuitSpec(kind="main" if spectator_state == "0" else "zz", protocol=protocol, basis=basis,\n                       t_us=timing.dt_to_us(t_dt), t_eff_us=t_eff, n_pulses=len(pulses),\n                       spectator_state=spectator_state)\n    return qc, spec\n\n\ndef build_readout_cal(n_qubits: int, state: str):\n    qc = QuantumCircuit(n_qubits, n_qubits, name=f"cal{state}")\n    if state == "1":\n        qc.x(range(n_qubits))\n    qc.barrier()\n    qc.measure(range(n_qubits), range(n_qubits))\n    return qc, CircuitSpec(kind=f"cal{state}")\n\n\ndef build_relay_reference(t_us: float, n_spectators: int, timing: Timing, target_state: str):\n    """Referencia pareada para relevos.\n    target_state=\'0\' -> objetivo inactivo (linea base de cada vecino a igual t).\n    target_state=\'1\' -> objetivo excitado |1> (caso mas sensible a transferencia por intercambio)."""\n    n = 1 + n_spectators\n    t_dt = timing.us_to_dt(t_us)\n    qc = QuantumCircuit(n, n, name=f"relay_t{target_state}_{t_dt}dt")\n    if target_state == "1":\n        qc.x(0)\n    qc.barrier()\n    if t_dt > 0:\n        qc.delay(t_dt, 0, unit="dt")\n    qc.barrier()\n    qc.measure(range(n), range(n))\n    kind = "idle" if target_state == "0" else "exc1"\n    return qc, CircuitSpec(kind=kind, t_us=timing.dt_to_us(t_dt), t_eff_us=timing.dt_to_us(t_dt))\n\n\ndef default_time_grid(t2_ref_us: float, fractions: Sequence[float] | None = None, t_max_us: float = 450.0,\n                      dense_us: Sequence[float] | None = None):\n    """Rejilla en dos tramos:\n    - tramo denso absoluto (paso <= 5 us hasta 40 us): resuelve la fase sin aliasing para |detuning| < ~100 kHz\n      (en v2/v3 el paso era de 26-130 us y el detuning real de ~27 kHz aparecia como ~54 kHz);\n    - tramo escalado a T2 calibrado (0.2x .. 1.6x): resuelve la envolvente y beta."""\n    if dense_us is None:\n        dense_us = [2, 5, 10, 15, 20, 25, 30, 40]\n    if fractions is None:\n        fractions = [0.2, 0.3, 0.45, 0.6, 0.8, 1.0, 1.3, 1.6]\n    grid = set(float(t) for t in dense_us)\n    grid |= set(round(min(f * t2_ref_us, t_max_us), 1) for f in fractions)\n    grid = sorted(g for g in grid if g > 0)\n    # elimina puntos casi duplicados (< 3 us)\n    out = []\n    for g in grid:\n        if not out or g - out[-1] >= 3.0:\n            out.append(g)\n    return out\n\n\ndef expected_parity_z(qc: QuantumCircuit) -> int:\n    """Utilidad para pruebas: numero de X/Y en el objetivo (debe ser par para volver a |+>)."""\n    return sum(1 for ins in qc.data if ins.operation.name in ("x", "y") and qc.find_bit(ins.qubits[0]).index == 0)\n\n\n__all__ = [\n    "PROTOCOLS", "BASES", "Timing", "timing_from_backend", "CircuitSpec",\n    "build_coherence_circuit", "build_readout_cal", "build_relay_reference", "default_time_grid",\n]\n_ = np  # numpy reservado para extensiones\n'
_FILES['backend_utils.py'] = '"""Seleccion de qubits, layout de espectadores, transpilacion verificada y estimacion de tiempo QPU."""\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass, asdict\nfrom typing import Dict, List, Optional\n\nimport numpy as np\n\nTWO_Q_GATES = {"cz", "ecr", "cx", "swap", "rzz", "iswap", "cy", "cp", "rzx"}\n\n\ndef get_token() -> str:\n    try:\n        from kaggle_secrets import UserSecretsClient  # type: ignore\n        t = UserSecretsClient().get_secret("IBM_QUANTUM_TOKEN")\n        if t:\n            return t.strip()\n    except Exception:\n        pass\n    t = os.environ.get("QISKIT_IBM_TOKEN") or os.environ.get("IBM_QUANTUM_TOKEN")\n    if not t:\n        raise RuntimeError("Configure el secreto IBM_QUANTUM_TOKEN (Kaggle) o la variable QISKIT_IBM_TOKEN")\n    return t.strip()\n\n\ndef get_service(channel: str = "ibm_cloud", instance: Optional[str] = "open-instance"):\n    """Crea el servicio sin escribir el token en disco (save_account no es necesario en cada corrida)."""\n    from qiskit_ibm_runtime import QiskitRuntimeService\n    kw = {"channel": channel, "token": get_token()}\n    if instance:\n        kw["instance"] = instance\n    return QiskitRuntimeService(**kw)\n\n\n@dataclass\nclass QubitInfo:\n    q: int\n    t1_us: float\n    t2_us: float\n    sx_err: float\n    ro_err: float\n    score: float\n\n\ndef _qubit_info(backend, q: int) -> Optional[QubitInfo]:\n    tgt = backend.target\n    try:\n        qp = tgt.qubit_properties[q]\n        t1 = (qp.t1 or 0.0) * 1e6\n        t2 = (qp.t2 or 0.0) * 1e6\n    except Exception:\n        return None\n    if t2 <= 0 or t1 <= 0:\n        return None\n\n    def err(op):\n        try:\n            e = tgt[op][(q,)].error\n            return float(e) if e is not None else None\n        except Exception:\n            return None\n\n    sx = err("sx") or err("x") or 1e-2\n    ro = err("measure") or 5e-2\n    # T2 > 2*T1 no es fisico: calibracion sospechosa, se recorta\n    t2_eff = min(t2, 2.0 * t1)\n    score = t2_eff / (1.0 + 50.0 * sx + 10.0 * ro)\n    return QubitInfo(q, t1, t2, sx, ro, score)\n\n\ndef rank_qubits(backend, top: int = 8) -> List[QubitInfo]:\n    rows = [r for q in range(backend.num_qubits) if (r := _qubit_info(backend, q))]\n    rows.sort(key=lambda r: -r.score)\n    return rows[:top] if top else rows\n\n\ndef bfs_distances(backend, source: int) -> Dict[int, int]:\n    adj: Dict[int, set] = {q: set() for q in range(backend.num_qubits)}\n    for a, b in backend.coupling_map.get_edges():\n        adj[a].add(b)\n        adj[b].add(a)\n    dist = {source: 0}\n    frontier = [source]\n    while frontier:\n        nxt = []\n        for u in frontier:\n            for v in adj[u]:\n                if v not in dist:\n                    dist[v] = dist[u] + 1\n                    nxt.append(v)\n        frontier = nxt\n    return dist\n\n\n@dataclass\nclass Layout:\n    target: int\n    target_t2_us: float\n    target_t1_us: float\n    neighbors: List[int]\n    control: int\n    control_distance: int\n\n    @property\n    def physical(self) -> List[int]:\n        return [self.target] + list(self.neighbors) + [self.control]\n\n    def to_dict(self):\n        d = asdict(self)\n        d["physical"] = self.physical\n        return d\n\n\ndef choose_layout(backend, force_target: Optional[int] = None, n_neighbors: int = 2,\n                  min_control_distance: int = 4) -> Layout:\n    ranked = rank_qubits(backend, top=0)\n    info = {r.q: r for r in ranked}\n    if force_target is not None:\n        tq = int(force_target)\n    else:\n        tq = ranked[0].q\n    ti = info.get(tq) or _qubit_info(backend, tq)\n    dist = bfs_distances(backend, tq)\n    nbs = sorted([q for q, d in dist.items() if d == 1], key=lambda q: -(info[q].score if q in info else 0))\n    if not nbs:\n        raise RuntimeError(f"q{tq} no tiene vecinos en el mapa de acoplamiento")\n    # Se miden TODOS los vecinos directos disponibles hasta n_neighbors (en heavy-hex son 2-3)\n    nbs = nbs[:n_neighbors]\n    # Control lejano: buena lectura (piso basal fiable) y lejos de objetivo y vecinos\n    far = [q for q, d in dist.items() if d >= min_control_distance and q in info]\n    if not far:\n        far = [q for q, d in dist.items() if d >= 2 and q in info]\n    far.sort(key=lambda q: (info[q].ro_err, -info[q].t1_us))\n    cq = far[0]\n    return Layout(tq, ti.t2_us, ti.t1_us, nbs, cq, dist[cq])\n\n\ndef transpile_locked(circuits, backend, layout: Layout, optimization_level: int = 1):\n    """Transpila con layout fijo y SIN enrutado. Devuelve (transpilados, reporte_de_relevos)."""\n    from qiskit.transpiler import generate_preset_pass_manager\n    pm = generate_preset_pass_manager(optimization_level=optimization_level, backend=backend,\n                                      initial_layout=layout.physical)\n    tcs = pm.run(list(circuits))\n    report = verify_no_relay(tcs, layout)\n    return tcs, report\n\n\ndef verify_no_relay(transpiled, layout: Layout) -> dict:\n    """Verificacion estructural de relevos.\n    - Cuenta compuertas de 2 qubits (un SWAP transpilado aparece como 3 CZ/ECR, no como \'swap\').\n    - Verifica layout inicial y final de cada qubit logico.\n    - Verifica que la medicion del bit 0 ocurre en el qubit fisico objetivo."""\n    issues = []\n    for i, tc in enumerate(transpiled):\n        ops = tc.count_ops()\n        n2q = sum(v for k, v in ops.items() if k in TWO_Q_GATES)\n        if n2q:\n            issues.append(f"circuito {i} ({tc.name}): {n2q} compuertas de 2 qubits")\n        lay = tc.layout\n        if lay is not None:\n            try:\n                init = lay.initial_index_layout(filter_ancillas=True)\n                fin = lay.final_index_layout(filter_ancillas=True)\n                if list(init) != layout.physical:\n                    issues.append(f"circuito {i}: layout inicial {list(init)} != {layout.physical}")\n                if list(fin) != layout.physical:\n                    issues.append(f"circuito {i}: layout final {list(fin)} != {layout.physical}")\n            except Exception as e:  # pragma: no cover\n                issues.append(f"circuito {i}: no se pudo leer layout ({e})")\n        for ins in tc.data:\n            if ins.operation.name == "measure":\n                pq = tc.find_bit(ins.qubits[0]).index\n                cb = tc.find_bit(ins.clbits[0]).index\n                if cb == 0 and pq != layout.target:\n                    issues.append(f"circuito {i}: bit 0 medido en q{pq} (esperado q{layout.target})")\n    return {"ok": not issues, "issues": issues, "n_circuits": len(transpiled)}\n\n\ndef estimate_qpu_seconds(specs, backend, rep_delay_s: Optional[float] = None) -> float:\n    """Estimacion aproximada: sum(shots * (t_efectivo + medicion + rep_delay)) + ~2 s de overhead."""\n    if rep_delay_s is None:\n        rep_delay_s = getattr(backend, "default_rep_delay", None) or 250e-6\n    try:\n        meas = backend.target["measure"][(0,)].duration or 1.6e-6\n    except Exception:\n        meas = 1.6e-6\n    total = sum(s.shots * (s.t_eff_us * 1e-6 + meas + rep_delay_s) for s in specs)\n    return float(total + 2.0)\n\n\n_ = np\n'
_FILES['analysis.py'] = '"""Analisis offline: mitigacion de lectura, coherencia en cuadratura, ajustes ponderados,\nestadistica de relevos (prueba pareada) y shift ZZ por espectadores."""\nfrom __future__ import annotations\n\nimport math\nfrom typing import Dict, List, Optional, Sequence\n\nimport numpy as np\nfrom scipy.optimize import curve_fit\nfrom scipy.stats import norm\n\n\n# ---------------------------------------------------------------- conteos\ndef counts_from_pub(pub) -> dict:\n    data = pub.data\n    for key in ("c", "meas"):\n        if hasattr(data, key):\n            return getattr(data, key).get_counts()\n    for key in data:\n        try:\n            return data[key].get_counts()\n        except Exception:\n            continue\n    return {}\n\n\ndef bit_counts(counts: dict, bit: int):\n    """(n0, n1) para el bit clasico `bit` (0 = el de la derecha)."""\n    n0 = n1 = 0\n    for k, v in counts.items():\n        b = str(k).replace(" ", "")[-1 - bit]\n        if b == "0":\n            n0 += int(v)\n        else:\n            n1 += int(v)\n    return n0, n1\n\n\n# ---------------------------------------------------------------- lectura\ndef readout_errors(cal0: dict, cal1: dict, n_bits: int):\n    """e0[b] = P(1|0), e1[b] = P(0|1) por qubit, desde los circuitos de calibracion del mismo job."""\n    e0, e1 = [], []\n    for b in range(n_bits):\n        a0, a1 = bit_counts(cal0, b)\n        c0, c1 = bit_counts(cal1, b)\n        e0.append(a1 / max(a0 + a1, 1))\n        e1.append(c0 / max(c0 + c1, 1))\n    return np.array(e0), np.array(e1)\n\n\ndef mitigated_expectation(counts: dict, bit: int, e0: float, e1: float):\n    """<Z> corregido por matriz de asignacion 1q y su error estandar."""\n    n0, n1 = bit_counts(counts, bit)\n    n = max(n0 + n1, 1)\n    pm1 = n1 / n\n    denom = max(1.0 - e0 - e1, 0.5)\n    p1 = float(np.clip((pm1 - e0) / denom, 0.0, 1.0))\n    ez = 1.0 - 2.0 * p1\n    se = 2.0 * math.sqrt(max(pm1 * (1 - pm1), 1.0 / n) / n) / denom\n    return ez, se\n\n\n# ---------------------------------------------------------------- coherencia\ndef coherence_from_xy(x, y, sx, sy):\n    """|rho01| = sqrt(<X>^2 + <Y>^2) con correccion de sesgo por ruido de disparo, y fase."""\n    x, y, sx, sy = map(lambda a: np.asarray(a, float), (x, y, sx, sy))\n    mag2 = x ** 2 + y ** 2 - (sx ** 2 + sy ** 2)\n    mag = np.sqrt(np.clip(mag2, 0.0, None))\n    se = np.sqrt((x * sx) ** 2 + (y * sy) ** 2) / np.maximum(mag, 1e-3)\n    se = np.maximum(se, np.sqrt(0.5 * (sx ** 2 + sy ** 2)))\n    phase = np.arctan2(y, x)\n    return mag, se, phase\n\n\ndef stretched(t, A, T2, beta):\n    return A * np.exp(-((np.maximum(t, 0.0) / T2) ** beta))\n\n\ndef fit_envelope(t, y, se, beta_fixed: Optional[float] = None, label: str = "") -> Optional[dict]:\n    """Ajuste ponderado A*exp(-(t/T2)^beta) con C = 0 (fisicamente, la coherencia decae a 0).\n    Fijar C elimina la degeneracion T2<->C que producia barras de error > valor en v1-v3.\n    El error se infla por sqrt(chi2_red) cuando hay deriva temporal (chi2_red > 1)."""\n    t, y, se = map(lambda a: np.asarray(a, float), (t, y, se))\n    ok = np.isfinite(y) & np.isfinite(se)\n    t, y, se = t[ok], y[ok], np.maximum(se[ok], 1e-3)\n    if len(t) < 3 or np.nanmax(y) < 0.05:\n        return None\n    # T0 inicial: primer cruce por 1/e\n    below = np.where(y < y[0] / math.e)[0]\n    T0 = float(t[below[0]]) if len(below) else float(t[-1] * 2)\n    try:\n        if beta_fixed is None:\n            f = stretched\n            p0, lo, hi = [min(y[0], 1.0), T0, 1.0], [0.0, 0.5, 0.5], [1.2, 20 * t[-1], 3.0]\n        else:\n            f = lambda tt, A, T2: stretched(tt, A, T2, beta_fixed)  # noqa: E731\n            p0, lo, hi = [min(y[0], 1.0), T0], [0.0, 0.5], [1.2, 20 * t[-1]]\n        popt, pcov = curve_fit(f, t, y, p0=p0, sigma=se, absolute_sigma=True, bounds=(lo, hi), maxfev=20000)\n    except Exception as e:\n        print(f"Ajuste {label} fallo: {e}")\n        return None\n    perr = np.sqrt(np.diag(pcov))\n    dof = max(len(t) - len(popt), 1)\n    chi2 = float(np.sum(((f(t, *popt) - y) / se) ** 2) / dof)\n    infl = math.sqrt(max(chi2, 1.0))\n    out = {"A": float(popt[0]), "T2": float(popt[1]), "T2_err": float(perr[1] * infl),\n           "beta": float(popt[2]) if beta_fixed is None else float(beta_fixed),\n           "beta_err": float(perr[2] * infl) if beta_fixed is None else 0.0,\n           "chi2_red": chi2, "n_points": int(len(t))}\n    out["reliable"] = bool(out["T2_err"] < 0.3 * out["T2"] and t[-1] > 0.7 * out["T2"])\n    return out\n\n\ndef detuning_from_phase(t, phase, mag, min_mag: float = 0.25, max_gap_us: float = 10.0):\n    """Frecuencia de detuning (kHz) por desenvolvimiento de fase en el tramo denso de la rejilla.\n    Solo se usan puntos consecutivos con separacion <= max_gap_us (sin aliasing si |f| < 1/(2*gap))."""\n    t, phase, mag = map(lambda a: np.asarray(a, float), (t, phase, mag))\n    sel = [0] if len(t) else []\n    for i in range(1, len(t)):\n        if mag[i] < min_mag or (t[i] - t[sel[-1]]) > max_gap_us:\n            break\n        sel.append(i)\n    if len(sel) < 3:\n        return None\n    ts, ph = t[sel], np.unwrap(phase[sel])\n    A = np.vstack([ts, np.ones_like(ts)]).T\n    coef, res, *_ = np.linalg.lstsq(A, ph, rcond=None)\n    slope = coef[0]\n    resid = ph - A @ coef\n    s2 = float(np.sum(resid ** 2) / max(len(ts) - 2, 1))\n    cov = s2 * np.linalg.inv(A.T @ A)\n    f_khz = slope / (2 * math.pi) * 1e3\n    f_err = math.sqrt(max(cov[0, 0], 0)) / (2 * math.pi) * 1e3\n    return {"f_kHz": float(f_khz), "f_err_kHz": float(f_err), "n_points": len(sel),\n            "t_max_us": float(ts[-1])}\n\n\ndef ratio_with_err(a, ea, b, eb):\n    r = a / b\n    return r, abs(r) * math.sqrt((ea / a) ** 2 + (eb / b) ** 2)\n\n\n# ---------------------------------------------------------------- relevos\ndef two_proportion_test(k1, n1, k0, n0):\n    """Diferencia de proporciones (p1 - p0), error, z, p-valor unilateral (p1 > p0)\n    y cota superior al 95%."""\n    p1, p0 = k1 / max(n1, 1), k0 / max(n0, 1)\n    p = (k1 + k0) / max(n1 + n0, 1)\n    se_pool = math.sqrt(max(p * (1 - p), 1e-12) * (1 / max(n1, 1) + 1 / max(n0, 1)))\n    se = math.sqrt(max(p1 * (1 - p1), 1 / max(n1, 1) ** 2) / max(n1, 1)\n                   + max(p0 * (1 - p0), 1 / max(n0, 1) ** 2) / max(n0, 1))\n    z = (p1 - p0) / se_pool if se_pool > 0 else 0.0\n    return {"delta": p1 - p0, "se": se, "z": z, "p_value": float(1 - norm.cdf(z)),\n            "upper95": (p1 - p0) + 1.645 * se, "p_active": p1, "p_ref": p0}\n\n\ndef relay_statistics(records: List[dict], neighbor_bits: Sequence[int], control_bit: int) -> dict:\n    """records: dicts con \'spec\' (CircuitSpec dict) y \'counts\'.\n    Compara P(1) de cada espectador cuando el objetivo esta activo (main, en |+>) o excitado (exc1)\n    contra la referencia con objetivo inactivo (idle), agregando todos los circuitos."""\n    def pool(kinds, bit):\n        k = n = 0\n        for r in records:\n            if r["spec"]["kind"] in kinds:\n                n0, n1 = bit_counts(r["counts"], bit)\n                k += n1\n                n += n0 + n1\n        return k, n\n\n    out = {}\n    for label, bit in [*[(f"vecino_bit{b}", b) for b in neighbor_bits], ("control", control_bit)]:\n        k_ref, n_ref = pool(("idle",), bit)\n        k_act, n_act = pool(("main",), bit)\n        k_exc, n_exc = pool(("exc1",), bit)\n        out[label] = {\n            "activo_vs_idle": two_proportion_test(k_act, n_act, k_ref, n_ref),\n            "excitado_vs_idle": two_proportion_test(k_exc, n_exc, k_ref, n_ref),\n            "n_ref": n_ref, "n_act": n_act, "n_exc": n_exc,\n        }\n    return out\n\n\ndef relay_verdict(stats: dict, alpha: float = 0.01, max_transfer: float = 0.005) -> dict:\n    """Hay relevo si algun vecino muestra exceso de P(1) significativo (p < alpha, con correccion de\n    Bonferroni) Y de magnitud relevante (> max_transfer). Si no, se reporta la cota superior."""\n    tests = [(lab, key, s[key]) for lab, s in stats.items() if lab.startswith("vecino")\n             for key in ("activo_vs_idle", "excitado_vs_idle")]\n    m = max(len(tests), 1)\n    flagged = [(lab, key, t) for lab, key, t in tests\n               if t["p_value"] < alpha / m and t["delta"] > max_transfer]\n    worst_upper = max((t["upper95"] for _, _, t in tests), default=float("nan"))\n    return {"relay_detected": bool(flagged), "flagged": [(a, b) for a, b, _ in flagged],\n            "worst_upper95": worst_upper, "bonferroni_m": m}\n\n\n__all__ = [n for n in dir() if not n.startswith("_")]\n'
_FILES['experiment.py'] = '"""Orquestacion: un unico job con barrido principal + referencias de relevo + ZZ + calibracion de lectura."""\nfrom __future__ import annotations\n\nimport json\nimport random\nimport time\nfrom dataclasses import dataclass, field, asdict\nfrom pathlib import Path\nfrom typing import List, Optional\n\nimport numpy as np\n\nfrom .circuits import (PROTOCOLS, BASES, build_coherence_circuit, build_readout_cal,\n                       build_relay_reference, default_time_grid, timing_from_backend)\nfrom .backend_utils import choose_layout, transpile_locked, estimate_qpu_seconds, Layout\nfrom . import analysis as A\n\n\n@dataclass\nclass Config:\n    backend_name: str = "ibm_fez"\n    force_target: Optional[int] = None       # p.ej. 142 para comparar con corridas previas\n    n_neighbors: int = 2\n    n_cpmg: int = 6\n    n_xy4_cycles: int = 2\n    shots_main: int = 1024                   # por base; X+Y => 2048 efectivos por punto\n    shots_ref: int = 2048                    # referencias de relevo (se agregan entre tiempos)\n    shots_cal: int = 4096\n    t2_fractions: Optional[List[float]] = None\n    t_max_us: float = 450.0\n    relay_fractions: List[float] = field(default_factory=lambda: [0.1, 0.5, 1.0])\n    zz_fractions: List[float] = field(default_factory=lambda: [0.05, 0.10, 0.20])\n    enable_zz: bool = True\n    shuffle_seed: Optional[int] = 1234       # orden aleatorio => la deriva no se correlaciona con t\n    optimization_level: int = 1\n    max_qpu_seconds: float = 240.0           # freno de seguridad para el plan Open (10 min/mes)\n\n\ndef build_batch(cfg: Config, backend, layout: Optional[Layout] = None):\n    layout = layout or choose_layout(backend, cfg.force_target, cfg.n_neighbors)\n    timing = timing_from_backend(backend)\n    n_spec = len(layout.physical) - 1\n    grid = default_time_grid(layout.target_t2_us, cfg.t2_fractions, cfg.t_max_us)\n    circs, specs = [], []\n\n    def add(qc, spec, shots):\n        spec.shots = shots\n        circs.append(qc)\n        specs.append(spec)\n\n    n = len(layout.physical)\n    for st in ("0", "1"):\n        add(*build_readout_cal(n, st), cfg.shots_cal)\n    for prot in PROTOCOLS:\n        for t in grid:\n            for b in BASES:\n                add(*build_coherence_circuit(prot, t, b, n_spec, timing, cfg.n_cpmg, cfg.n_xy4_cycles,\n                                             "0", cfg.n_neighbors), cfg.shots_main)\n    for f in cfg.relay_fractions:\n        t = f * layout.target_t2_us\n        add(*build_relay_reference(t, n_spec, timing, "0"), cfg.shots_ref)\n        add(*build_relay_reference(t, n_spec, timing, "1"), cfg.shots_ref)\n    if cfg.enable_zz:\n        # se usan puntos EXACTOS de la rejilla principal para comparar fase a igual t\n        zz_t = sorted(set(min(grid, key=lambda g: abs(g - f * layout.target_t2_us)) for f in cfg.zz_fractions))\n        for prot in ("free", "cpmg"):\n            for t in zz_t:\n                for b in BASES:\n                    add(*build_coherence_circuit(prot, t, b, n_spec, timing, cfg.n_cpmg, cfg.n_xy4_cycles,\n                                                 "1", cfg.n_neighbors), cfg.shots_main)\n    order = list(range(len(circs)))\n    if cfg.shuffle_seed is not None:\n        random.Random(cfg.shuffle_seed).shuffle(order)\n    circs = [circs[i] for i in order]\n    specs = [specs[i] for i in order]\n    return circs, specs, layout, timing, grid\n\n\ndef run_batch(cfg: Config, backend, circs, specs, layout, out_dir: Path, mode: str = "hardware"):\n    """Transpila con layout bloqueado, verifica relevos estructurales y envia UN job.\n    mode: \'hardware\' | \'sim\' (backend falso local)."""\n    from qiskit_ibm_runtime import SamplerV2\n    tcs, report = transpile_locked(circs, backend, layout, cfg.optimization_level)\n    if not report["ok"]:\n        raise RuntimeError("Verificacion estructural fallo (posible relevo):\\n" + "\\n".join(report["issues"][:10]))\n    est = estimate_qpu_seconds(specs, backend)\n    print(f"Circuitos: {len(tcs)} | shots totales: {sum(s.shots for s in specs)} | QPU estimado ~{est:.0f} s")\n    if mode == "hardware" and est > cfg.max_qpu_seconds:\n        raise RuntimeError(f"Estimacion {est:.0f}s > max_qpu_seconds={cfg.max_qpu_seconds}. Reduce shots o puntos.")\n    sampler = SamplerV2(mode=backend)\n    pubs = [(tc, None, s.shots) for tc, s in zip(tcs, specs)]\n    job = sampler.run(pubs)\n    job_id = job.job_id()\n    print("Job ID:", job_id)\n    manifest = {"job_id": job_id, "backend": backend.name, "layout": layout.to_dict(),\n                "config": asdict(cfg), "specs": [s.to_dict() for s in specs],\n                "structural_check": report, "qpu_estimate_s": est, "created": time.strftime("%Y-%m-%d %H:%M:%S")}\n    out_dir.mkdir(parents=True, exist_ok=True)\n    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=1))\n    t0 = time.time()\n    result = job.result()\n    print(f"Job completado en {time.time() - t0:.0f} s (incluye cola)")\n    counts = [A.counts_from_pub(result[i]) for i in range(len(specs))]\n    save_counts(out_dir, manifest, counts)\n    return manifest, counts\n\n\ndef save_counts(out_dir: Path, manifest: dict, counts: list):\n    raw = dict(manifest)\n    raw["counts"] = counts\n    (out_dir / "raw_counts.json").write_text(json.dumps(raw))\n\n\ndef reload_from_job(service, job_id: str, manifest_path: Path):\n    """Reanalisis sin volver a hacer cola: recupera el resultado de un job ya ejecutado."""\n    manifest = json.loads(Path(manifest_path).read_text())\n    res = service.job(job_id).result()\n    counts = [A.counts_from_pub(res[i]) for i in range(len(manifest["specs"]))]\n    return manifest, counts\n\n\n# ------------------------------------------------------------------ analisis completo\ndef analyze(manifest: dict, counts: list) -> dict:\n    specs = manifest["specs"]\n    lay = manifest["layout"]\n    n_bits = len(lay["physical"])\n    recs = [{"spec": s, "counts": c} for s, c in zip(specs, counts)]\n    cal0 = next(r["counts"] for r in recs if r["spec"]["kind"] == "cal0")\n    cal1 = next(r["counts"] for r in recs if r["spec"]["kind"] == "cal1")\n    e0, e1 = A.readout_errors(cal0, cal1, n_bits)\n\n    def series(kind, prot):\n        pts = {}\n        for r in recs:\n            s = r["spec"]\n            if s["kind"] == kind and s["protocol"] == prot:\n                ez, se = A.mitigated_expectation(r["counts"], 0, e0[0], e1[0])\n                pts.setdefault(round(s["t_us"], 4), {"t_eff": s["t_eff_us"]})[s["basis"]] = (ez, se)\n        ts = sorted(k for k, v in pts.items() if "X" in v and "Y" in v)\n        t = np.array(ts)\n        teff = np.array([pts[k]["t_eff"] for k in ts])\n        x = np.array([pts[k]["X"][0] for k in ts]); sx = np.array([pts[k]["X"][1] for k in ts])\n        y = np.array([pts[k]["Y"][0] for k in ts]); sy = np.array([pts[k]["Y"][1] for k in ts])\n        mag, smag, ph = A.coherence_from_xy(x, y, sx, sy)\n        return {"t": t, "t_eff": teff, "X": x, "sX": sx, "Y": y, "sY": sy, "mag": mag, "smag": smag, "phase": ph}\n\n    out = {"readout": {"e0": e0.tolist(), "e1": e1.tolist()}, "series": {}, "fits": {}, "fits_exp": {}}\n    for p in PROTOCOLS:\n        s = series("main", p)\n        out["series"][p] = s\n        out["fits"][p] = A.fit_envelope(s["t_eff"], s["mag"], s["smag"], label=p)\n        out["fits_exp"][p] = A.fit_envelope(s["t_eff"], s["mag"], s["smag"], beta_fixed=1.0, label=p + "_exp")\n    out["detuning"] = A.detuning_from_phase(out["series"]["free"]["t"], out["series"]["free"]["phase"],\n                                           out["series"]["free"]["mag"])\n    # Extension de coherencia (con barras de error)\n    ext = {}\n    ff = out["fits"]["free"]\n    for p in ("cpmg", "xy4"):\n        fp = out["fits"][p]\n        if ff and fp:\n            r, er = A.ratio_with_err(fp["T2"], fp["T2_err"], ff["T2"], ff["T2_err"])\n            ext[p] = {"ratio": r, "err": er, "significant": bool(r - 1 > 2 * er)}\n    out["extension"] = ext\n    # Relevos\n    nb_bits = list(range(1, 1 + len(lay["neighbors"])))\n    rs = A.relay_statistics(recs, nb_bits, n_bits - 1)\n    out["relay"] = rs\n    out["relay_verdict"] = A.relay_verdict(rs)\n    # ZZ: comparar fase y coherencia con vecinos en |1> vs |0> a los mismos t\n    zz = {}\n    if any(r["spec"]["kind"] == "zz" for r in recs):\n        for p in ("free", "cpmg"):\n            s1 = series("zz", p)\n            s0 = out["series"][p]\n            rows = []\n            for i, t in enumerate(s1["t"]):\n                j = np.argmin(np.abs(s0["t"] - t))\n                if abs(s0["t"][j] - t) > 1e-3:\n                    continue\n                dphi = float(np.angle(np.exp(1j * (s1["phase"][i] - s0["phase"][j]))))\n                rows.append({"t": float(t), "dphi": dphi, "mag1": float(s1["mag"][i]), "mag0": float(s0["mag"][j])})\n            if rows:\n                tt = np.array([r["t"] for r in rows]); dp = np.array([r["dphi"] for r in rows])\n                slope = float(np.sum(tt * dp) / np.sum(tt * tt))\n                zz[p] = {"rows": rows, "zz_kHz": slope / (2 * np.pi) * 1e3}\n    out["zz"] = zz\n    return out\n\n\ndef to_jsonable(o):\n    if isinstance(o, dict):\n        return {k: to_jsonable(v) for k, v in o.items()}\n    if isinstance(o, (list, tuple)):\n        return [to_jsonable(v) for v in o]\n    if isinstance(o, np.ndarray):\n        return o.tolist()\n    if isinstance(o, (np.floating, np.integer, np.bool_)):\n        return o.item()\n    return o\n\n\ndef summarize(res: dict, manifest: dict) -> str:\n    L = []\n    lay = manifest["layout"]\n    L.append(f"Backend={manifest[\'backend\']}  objetivo=q{lay[\'target\']} (T2 cal={lay[\'target_t2_us\']:.1f} us, "\n             f"T1 cal={lay[\'target_t1_us\']:.1f} us)  vecinos={lay[\'neighbors\']}  control=q{lay[\'control\']}  "\n             f"job={manifest[\'job_id\']}")\n    L.append(f"Error de lectura objetivo: P(1|0)={res[\'readout\'][\'e0\'][0]:.4f}  P(0|1)={res[\'readout\'][\'e1\'][0]:.4f}")\n    for p in PROTOCOLS:\n        f = res["fits"][p]\n        if f:\n            tag = "" if f["reliable"] else "  [baja confianza]"\n            L.append(f"T2[{p:>4}] = {f[\'T2\']:7.1f} +/- {f[\'T2_err\']:5.1f} us  beta={f[\'beta\']:.2f}+/-{f[\'beta_err\']:.2f}"\n                     f"  chi2_red={f[\'chi2_red\']:.1f}{tag}")\n        else:\n            L.append(f"T2[{p:>4}] = sin ajuste")\n    d = res.get("detuning")\n    if d:\n        L.append(f"Detuning (fase desenvuelta, {d[\'n_points\']} pts hasta {d[\'t_max_us\']:.0f} us): "\n                 f"{d[\'f_kHz\']:+.2f} +/- {d[\'f_err_kHz\']:.2f} kHz")\n    for p, e in res["extension"].items():\n        L.append(f"Extension {p}/libre = {e[\'ratio\']:.2f} +/- {e[\'err\']:.2f}  -> "\n                 f"{\'significativa (>2 sigma)\' if e[\'significant\'] else \'NO significativa\'}")\n    rv = res["relay_verdict"]\n    L.append("Relevos: " + ("DETECTADO en " + str(rv["flagged"]) if rv["relay_detected"] else\n                             f"no detectado; cota superior 95% de transferencia a vecinos = {rv[\'worst_upper95\']*100:.3f} %"))\n    for p, z in res.get("zz", {}).items():\n        L.append(f"ZZ efectivo (vecinos en |1>) con {p}: {z[\'zz_kHz\']:+.2f} kHz")\n    return "\\n".join(L)\n'
_FILES['plots.py'] = '"""Graficas del analisis."""\nfrom __future__ import annotations\n\nimport numpy as np\nimport matplotlib.pyplot as plt\n\nfrom .analysis import stretched\nfrom .circuits import PROTOCOLS\n\nCOLORS = {"free": "#c0392b", "cpmg": "#2471a3", "xy4": "#1e8449"}\nLABELS = {"free": "Libre (Ramsey)", "cpmg": "CPMG", "xy4": "XY4"}\n\n\ndef plot_coherence(res, manifest, path=None):\n    lay = manifest["layout"]\n    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))\n    ax = axes[0]\n    for p in PROTOCOLS:\n        s = res["series"][p]\n        ax.errorbar(s["t_eff"], s["mag"], yerr=s["smag"], fmt="o", ms=4, color=COLORS[p], label=LABELS[p], capsize=2)\n        f = res["fits"][p]\n        if f:\n            tt = np.linspace(0, max(s["t_eff"]) * 1.05, 300)\n            ax.plot(tt, stretched(tt, f["A"], f["T2"], f["beta"]), "-", color=COLORS[p], alpha=0.8,\n                    label=f"{LABELS[p]}: T2={f[\'T2\']:.0f}±{f[\'T2_err\']:.0f} µs, β={f[\'beta\']:.2f}")\n    ax.set_xlabel("t efectivo (µs)"); ax.set_ylabel("|ρ01| = √(⟨X⟩²+⟨Y⟩²)")\n    ax.set_title(f"Coherencia (cuadratura, lectura mitigada) — {manifest[\'backend\']} q{lay[\'target\']}")\n    ax.set_ylim(-0.03, 1.05); ax.grid(alpha=0.3); ax.legend(fontsize=7.5)\n\n    ax = axes[1]\n    s = res["series"]["free"]\n    ax.errorbar(s["t"], s["X"], yerr=s["sX"], fmt="o-", ms=3, color="#c0392b", label="⟨X⟩ libre")\n    ax.errorbar(s["t"], s["Y"], yerr=s["sY"], fmt="s-", ms=3, color="#7d3c98", label="⟨Y⟩ libre")\n    ax.plot(s["t"], s["mag"], "k--", lw=1, label="|ρ01| libre")\n    ax.axhline(0, color="gray", lw=0.8)\n    d = res.get("detuning")\n    ttl = "Libre: X/Y (el cruce por cero es fase, no pérdida)"\n    if d:\n        ttl += f"\\nδf = {d[\'f_kHz\']:+.1f} ± {d[\'f_err_kHz\']:.1f} kHz"\n    ax.set_title(ttl, fontsize=10); ax.set_xlabel("t (µs)"); ax.grid(alpha=0.3); ax.legend(fontsize=8)\n    fig.tight_layout()\n    if path:\n        fig.savefig(path, dpi=140)\n    return fig\n\n\ndef plot_relay(res, manifest, path=None):\n    rs = res["relay"]\n    labs = list(rs.keys())\n    lay = manifest["layout"]\n    names = [f"q{q}" for q in lay["neighbors"]] + [f"q{lay[\'control\']} (control)"]\n    fig, ax = plt.subplots(figsize=(8, 4.2))\n    x = np.arange(len(labs))\n    for k, (key, lab, col) in enumerate([("activo_vs_idle", "objetivo en |+> (DD/libre) − idle", "#2471a3"),\n                                         ("excitado_vs_idle", "objetivo en |1> − idle", "#c0392b")]):\n        d = [rs[l][key]["delta"] * 100 for l in labs]\n        e = [rs[l][key]["se"] * 100 for l in labs]\n        ax.bar(x + (k - 0.5) * 0.36, d, 0.36, yerr=e, capsize=3, color=col, label=lab)\n    ax.axhline(0, color="k", lw=0.8)\n    ax.set_xticks(x); ax.set_xticklabels(names)\n    ax.set_ylabel("Δ P(1) espectador (%)")\n    v = res["relay_verdict"]\n    ax.set_title("Relevos: exceso de población en espectadores (prueba pareada)\\n"\n                 + ("RELEVO DETECTADO" if v["relay_detected"] else f"sin relevo; cota 95% = {v[\'worst_upper95\']*100:.3f}%"),\n                 fontsize=10)\n    ax.grid(alpha=0.3, axis="y"); ax.legend(fontsize=8)\n    fig.tight_layout()\n    if path:\n        fig.savefig(path, dpi=140)\n    return fig\n'
_FILES['sentinel.py'] = '"""Centinela: sondeo previo (pre-flight) de pocos segundos de QPU para decidir, justo antes de\nun trabajo caro, que qubit usar, si activar DD y que correccion de fase aplicar.\n\nMotivacion fisica (corridas v4 en ibm_fez q142, 23-sep-2026):\n- El qubit alterna entre dos estados en escala de minutos: T2* ~155 us y ~40 us, con\n  detuning -27.0 kHz y -21..-25 kHz. Es la firma de un sistema de dos niveles (TLS)\n  fluctuante acoplado al qubit: saltos telegraficos de frecuencia correlacionados con desfase.\n- La calibracion publicada (T2 = 265 us) se actualiza ~1 vez al dia y no ve esos saltos.\n- La frecuencia es la variable mas barata de medir y cambia con el estado: se usa como\n  "huella" junto con T2*.\n\nDiseno del sondeo (un solo job, todos los candidatos EN PARALELO en los mismos circuitos):\n- Ramsey en cuadratura (bases X e Y) en un tramo denso (<= 10 us de paso, sin alias hasta 50 kHz)\n  mas 2 puntos largos para T2*.\n- 2 puntos CPMG (eco multiple) para estimar cuanto recupera el DD en el estado actual.\n- 2 puntos de T1 para separar desfase puro: Gamma_phi = 1/T2* - 1/(2 T1).\n- Calibracion de lectura propia.\n~28 circuitos x 512 shots ~ 6-8 s de QPU para K candidatos a la vez.\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport time\nfrom dataclasses import dataclass, field, asdict\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence\n\nimport numpy as np\nfrom qiskit import QuantumCircuit\n\nfrom .circuits import Timing, CircuitSpec, timing_from_backend, _dd_block, _pulse_sequence\nfrom .backend_utils import rank_qubits, bfs_distances, _qubit_info\nfrom . import analysis as A\n\nUSD_PER_MIN_PAYG = 96.0   # IBM Pay-As-You-Go (precio publico de lista, 2026)\n\n\n# ------------------------------------------------------------------ configuracion\n@dataclass\nclass ProbeConfig:\n    ramsey_dense_us: List[float] = field(default_factory=lambda: [2, 5, 10, 15, 20, 25, 30, 40])\n    ramsey_long_us: List[float] = field(default_factory=lambda: [80, 160])\n    echo_us: List[float] = field(default_factory=lambda: [80, 160])\n    t1_us: List[float] = field(default_factory=lambda: [80, 160])\n    n_cpmg: int = 6\n    shots: int = 512\n    shots_cal: int = 2048\n    n_candidates: int = 4\n    min_separation: int = 3          # distancia minima en el grafo entre candidatos (evita ZZ entre ellos)\n    optimization_level: int = 1\n\n\ndef choose_candidates(backend, k: int = 4, min_separation: int = 3, must_include: Sequence[int] = ()) -> List[int]:\n    """Top-k qubits por calibracion con separacion minima en el grafo de acoplamiento."""\n    chosen = [int(q) for q in must_include]\n    for r in rank_qubits(backend, top=10 * k + len(chosen)):\n        if len(chosen) >= k:\n            break\n        if r.q in chosen:\n            continue\n        d = bfs_distances(backend, r.q)\n        if all(d.get(c, 99) >= min_separation for c in chosen):\n            chosen.append(r.q)\n    return chosen\n\n\n# ------------------------------------------------------------------ circuitos\ndef _ramsey(n: int, t_dt: int, basis: str, timing: Timing, pulses: Sequence[str] = ()):\n    qc = QuantumCircuit(n, n)\n    qc.h(range(n))\n    qc.barrier()\n    for q in range(n):\n        _dd_block(qc, q, t_dt, list(pulses), timing)\n    qc.barrier()\n    if basis == "Y":\n        qc.sdg(range(n))\n    qc.h(range(n))\n    qc.measure(range(n), range(n))\n    return qc\n\n\ndef _t1(n: int, t_dt: int):\n    qc = QuantumCircuit(n, n)\n    qc.x(range(n))\n    qc.barrier()\n    if t_dt > 0:\n        qc.delay(t_dt, range(n), unit="dt")\n    qc.barrier()\n    qc.measure(range(n), range(n))\n    return qc\n\n\ndef build_probe(n_qubits: int, timing: Timing, pc: ProbeConfig = ProbeConfig()):\n    """Circuitos logicos del sondeo sobre n_qubits candidatos en paralelo."""\n    circs, specs = [], []\n\n    def add(qc, spec, shots):\n        spec.shots = shots\n        qc.name = f"{spec.kind}_{spec.protocol}_{spec.basis}_{spec.t_us:.1f}"\n        circs.append(qc)\n        specs.append(spec)\n\n    for st in ("0", "1"):\n        qc = QuantumCircuit(n_qubits, n_qubits)\n        if st == "1":\n            qc.x(range(n_qubits))\n        qc.measure(range(n_qubits), range(n_qubits))\n        add(qc, CircuitSpec(kind=f"cal{st}"), pc.shots_cal)\n    for t in list(pc.ramsey_dense_us) + list(pc.ramsey_long_us):\n        t_dt = timing.us_to_dt(t)\n        for b in ("X", "Y"):\n            add(_ramsey(n_qubits, t_dt, b, timing), CircuitSpec(kind="ramsey", protocol="free", basis=b,\n                t_us=timing.dt_to_us(t_dt), t_eff_us=timing.dt_to_us(t_dt)), pc.shots)\n    pulses = _pulse_sequence("cpmg", pc.n_cpmg, 1)\n    for t in pc.echo_us:\n        t_dt = timing.us_to_dt(t)\n        for b in ("X", "Y"):\n            add(_ramsey(n_qubits, t_dt, b, timing, pulses), CircuitSpec(kind="echo", protocol="cpmg", basis=b,\n                t_us=timing.dt_to_us(t_dt), t_eff_us=timing.dt_to_us(t_dt + len(pulses) * timing.x_duration_dt),\n                n_pulses=len(pulses)), pc.shots)\n    for t in pc.t1_us:\n        t_dt = timing.us_to_dt(t)\n        add(_t1(n_qubits, t_dt), CircuitSpec(kind="t1", basis="Z", t_us=timing.dt_to_us(t_dt),\n                                             t_eff_us=timing.dt_to_us(t_dt)), pc.shots)\n    return circs, specs\n\n\ndef probe_cost(specs, backend=None, usd_per_min: float = USD_PER_MIN_PAYG, rep_delay_s: float = 250e-6,\n               meas_s: float = 1.6e-6):\n    secs = sum(s.shots * (s.t_eff_us * 1e-6 + meas_s + rep_delay_s) for s in specs) + 2.0\n    return {"qpu_s": float(secs), "usd": float(secs / 60.0 * usd_per_min)}\n\n\ndef run_probe(backend, candidates: Sequence[int], out_dir: Path, pc: ProbeConfig = ProbeConfig(),\n              max_qpu_seconds: float = 20.0, mode: str = "hardware"):\n    """Transpila con layout fijo [candidatos], verifica que no haya 2q/SWAP y envia UN job."""\n    from qiskit import transpile\n    from qiskit_ibm_runtime import SamplerV2\n    timing = timing_from_backend(backend)\n    circs, specs = build_probe(len(candidates), timing, pc)\n    tcs = transpile(circs, backend, initial_layout=list(candidates), optimization_level=pc.optimization_level,\n                    scheduling_method=None)\n    for tc in tcs:\n        n2q = sum(1 for ins in tc.data if ins.operation.num_qubits == 2 and ins.operation.name != "barrier")\n        if n2q:\n            raise RuntimeError(f"{tc.name}: {n2q} compuertas de 2 qubits (posible SWAP)")\n    cost = probe_cost(specs)\n    if mode == "hardware" and cost["qpu_s"] > max_qpu_seconds:\n        raise RuntimeError(f"Sondeo estimado {cost[\'qpu_s\']:.1f}s > {max_qpu_seconds}s")\n    job = SamplerV2(mode=backend).run([(tc, None, s.shots) for tc, s in zip(tcs, specs)])\n    manifest = {"job_id": job.job_id(), "backend": backend.name, "candidates": list(map(int, candidates)),\n                "specs": [s.to_dict() for s in specs], "probe_config": asdict(pc), "cost": cost,\n                "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}\n    res = job.result()\n    counts = [A.counts_from_pub(res[i]) for i in range(len(specs))]\n    out_dir.mkdir(parents=True, exist_ok=True)\n    (out_dir / "probe_raw.json").write_text(json.dumps({**manifest, "counts": counts}))\n    return manifest, counts\n\n\n# ------------------------------------------------------------------ analisis\ndef _fit_fixed_amp(t, y, se, amp):\n    """y = amp * exp(-t/T): estimacion de T por minimos cuadrados ponderados en log."""\n    t, y, se = map(lambda a: np.asarray(a, float), (t, y, se))\n    ok = (y > 0.02) & np.isfinite(y)\n    if ok.sum() == 0:\n        return None, None\n    rates, w = [], []\n    for ti, yi, si in zip(t[ok], y[ok], se[ok]):\n        r = -math.log(min(yi / amp, 0.999)) / ti\n        er = si / yi / ti\n        rates.append(r); w.append(1 / max(er, 1e-9) ** 2)\n    rates, w = np.array(rates), np.array(w)\n    r = float(np.sum(w * rates) / np.sum(w)); er = float(math.sqrt(1 / np.sum(w)))\n    if r <= 0:\n        return None, None\n    return 1 / r, er / r ** 2\n\n\ndef analyze_probe(manifest: dict, counts: list) -> Dict[int, dict]:\n    specs, cands = manifest["specs"], manifest["candidates"]\n    recs = list(zip(specs, counts))\n    cal0 = next(c for s, c in recs if s["kind"] == "cal0")\n    cal1 = next(c for s, c in recs if s["kind"] == "cal1")\n    e0, e1 = A.readout_errors(cal0, cal1, len(cands))\n    out = {}\n    for b, q in enumerate(cands):\n        def xy(kind):\n            ts = sorted({s["t_us"] for s, _ in recs if s["kind"] == kind})\n            X, Y, SX, SY = [], [], [], []\n            for t in ts:\n                cx = next(c for s, c in recs if s["kind"] == kind and s["t_us"] == t and s["basis"] == "X")\n                cy = next(c for s, c in recs if s["kind"] == kind and s["t_us"] == t and s["basis"] == "Y")\n                x, sx = A.mitigated_expectation(cx, b, e0[b], e1[b])\n                y, sy = A.mitigated_expectation(cy, b, e0[b], e1[b])\n                X.append(x); Y.append(y); SX.append(sx); SY.append(sy)\n            mag, se, ph = A.coherence_from_xy(X, Y, SX, SY)\n            return np.array(ts), mag, se, ph\n        tr, mr, sr, pr = xy("ramsey")\n        det = A.detuning_from_phase(tr, pr, mr)\n        fit = A.fit_envelope(tr, mr, sr, beta_fixed=1.0, label=f"q{q}")\n        amp = fit["A"] if fit else float(mr[0])\n        te, me, se_, _ = xy("echo")\n        T2e, T2e_err = _fit_fixed_amp(te, me, se_, amp)\n        # T1: P1(t) mitigada, amplitud ~1 tras mitigar\n        t1s, p1s, s1s = [], [], []\n        for s, c in recs:\n            if s["kind"] == "t1":\n                z, sz = A.mitigated_expectation(c, b, e0[b], e1[b])\n                t1s.append(s["t_us"]); p1s.append((1 - z) / 2); s1s.append(sz / 2)\n        T1, T1_err = _fit_fixed_amp(t1s, p1s, s1s, 1.0)\n        r = {"q": int(q), "readout_e0": float(e0[b]), "readout_e1": float(e1[b]),\n             "detuning_kHz": det["f_kHz"] if det else None, "detuning_err_kHz": det["f_err_kHz"] if det else None,\n             "T2star_us": fit["T2"] if fit else None, "T2star_err_us": fit["T2_err"] if fit else None,\n             "T2echo_us": T2e, "T2echo_err_us": T2e_err, "T1_us": T1, "T1_err_us": T1_err}\n        if r["T2star_us"] and T1:\n            r["Gamma_phi_1_ms"] = 1e3 * (1 / r["T2star_us"] - 1 / (2 * T1))\n        out[int(q)] = r\n    return out\n\n\n# ------------------------------------------------------------------ bitacora y decision\nclass Bitacora:\n    """Historial JSONL por (backend, qubit). Es la base de datos del producto: con el tiempo da la\n    estadistica de estados (fraccion de tiempo degradado, tiempos de permanencia) por qubit."""\n\n    def __init__(self, path: Path):\n        self.path = Path(path)\n\n    def append(self, backend: str, probe: Dict[int, dict], verdicts: Dict[int, dict], when: Optional[str] = None):\n        when = when or time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        with self.path.open("a") as f:\n            for q, r in probe.items():\n                f.write(json.dumps({"t": when, "backend": backend, **r, "estado": verdicts[q]["estado"]}) + "\\n")\n\n    def history(self, backend: str, q: int) -> List[dict]:\n        if not self.path.exists():\n            return []\n        rows = [json.loads(l) for l in self.path.read_text().splitlines() if l.strip()]\n        return [r for r in rows if r["backend"] == backend and r["q"] == q]\n\n\ndef reference_state(history: List[dict], T2_cal_us: Optional[float] = None) -> dict:\n    """Huella del estado \'bueno\': mediana de las entradas estables (o de todas si no hay etiqueta)."""\n    good = [h for h in history if h.get("estado") == "estable" and h.get("T2star_us")]\n    pool = good or [h for h in history if h.get("T2star_us")]\n    if pool:\n        t2 = float(np.median([h["T2star_us"] for h in pool]))\n        f = [h["detuning_kHz"] for h in pool if h.get("detuning_kHz") is not None]\n        return {"T2star_us": t2, "detuning_kHz": float(np.median(f)) if f else None, "n": len(pool)}\n    # sin historial: T2* ~ 0.5 x T2 calibrado (en q142: 155/265 = 0.59)\n    return {"T2star_us": 0.5 * T2_cal_us if T2_cal_us else None, "detuning_kHz": None, "n": 0}\n\n\ndef evaluate(r: dict, ref: dict, min_ratio: float = 0.6, max_df_kHz: float = 1.0) -> dict:\n    """Estado del qubit y accion recomendada."""\n    why = []\n    estado = "estable"\n    if r.get("T2star_us") is None:\n        return {"estado": "sin_datos", "accion": "re-sondear", "motivo": ["ajuste T2* fallo"]}\n    if ref.get("T2star_us") and r["T2star_us"] < min_ratio * ref["T2star_us"]:\n        estado = "degradado"; why.append(f"T2* {r[\'T2star_us\']:.0f} us < {min_ratio:.0%} de referencia {ref[\'T2star_us\']:.0f} us")\n    if ref.get("detuning_kHz") is not None and r.get("detuning_kHz") is not None:\n        tol = max(max_df_kHz, 3 * (r.get("detuning_err_kHz") or 0))\n        if abs(r["detuning_kHz"] - ref["detuning_kHz"]) > tol:\n            estado = "degradado"; why.append(f"salto de frecuencia {r[\'detuning_kHz\'] - ref[\'detuning_kHz\']:+.1f} kHz")\n    if estado == "estable":\n        accion = "ejecutar"\n    else:\n        accion = "cambiar de qubit o esperar y re-sondear; si se ejecuta, activar DD"\n    gain = None\n    if r.get("T2echo_us") and r.get("T2star_us"):\n        gain = r["T2echo_us"] / r["T2star_us"]\n    return {"estado": estado, "accion": accion, "motivo": why or ["dentro de la huella de referencia"],\n            "ganancia_dd_estimada": gain,\n            "correccion_fase_kHz": r.get("detuning_kHz")}\n\n\ndef rank_live(probe: Dict[int, dict], verdicts: Dict[int, dict]) -> List[int]:\n    """Orden por T2* medido ahora (estables primero)."""\n    return sorted(probe, key=lambda q: (verdicts[q]["estado"] != "estable", -(probe[q].get("T2star_us") or 0)))\n\n\n# ------------------------------------------------------------------ correccion de fase (Z virtual)\ndef compensate_detuning(qc: QuantumCircuit, detuning_kHz: Dict[int, float], dt: float) -> QuantumCircuit:\n    """Inserta rz(-2*pi*f*tau) despues de cada delay del qubit q (indice del circuito).\n    En el modelo de detuning estatico cancela exactamente la fase acumulada en cada tramo, con o sin\n    pulsos DD intermedios (ambos son rotaciones en Z que conmutan con la evolucion libre).\n    Las rz son compuertas virtuales en IBM: costo cero en tiempo y error."""\n    out = qc.copy_empty_like()\n    for ins in qc.data:\n        out.append(ins.operation, ins.qubits, ins.clbits)\n        if ins.operation.name == "delay":\n            dur = ins.operation.duration\n            unit = ins.operation.unit\n            tau = dur * dt if unit == "dt" else dur * {"s": 1, "ms": 1e-3, "us": 1e-6, "ns": 1e-9}[unit]\n            for qb in ins.qubits:\n                q = qc.find_bit(qb).index\n                if q in detuning_kHz and detuning_kHz[q]:\n                    out.rz(-2 * math.pi * detuning_kHz[q] * 1e3 * tau, q)\n    return out\n\n\ndef summarize_probe(backend_name: str, probe: Dict[int, dict], verdicts: Dict[int, dict], cost: dict) -> str:\n    L = [f"Centinela | backend={backend_name} | costo ~{cost[\'qpu_s\']:.1f} s QPU (~USD {cost[\'usd\']:.2f} a precio PAYG)"]\n    L.append(f"{\'q\':>5} {\'T2*\':>8} {\'T2 eco\':>8} {\'T1\':>8} {\'detuning\':>10} {\'estado\':>10}  accion")\n    for q in rank_live(probe, verdicts):\n        r, v = probe[q], verdicts[q]\n        fmt = lambda x: f"{x:8.1f}" if x else "     n/a"  # noqa: E731\n        det = f"{r[\'detuning_kHz\']:+9.2f}k" if r.get("detuning_kHz") is not None else "       n/a"\n        L.append(f"{q:>5} {fmt(r.get(\'T2star_us\'))} {fmt(r.get(\'T2echo_us\'))} {fmt(r.get(\'T1_us\'))} {det} {v[\'estado\']:>10}  {v[\'accion\']}")\n    return "\\n".join(L)\n\n\n__all__ = ["ProbeConfig", "choose_candidates", "build_probe", "probe_cost", "run_probe", "analyze_probe",\n           "Bitacora", "reference_state", "evaluate", "rank_live", "compensate_detuning", "summarize_probe",\n           "USD_PER_MIN_PAYG"]\n_ = _qubit_info\n'
pkg = pathlib.Path("ddqpu"); pkg.mkdir(exist_ok=True)
for n, s in _FILES.items():
    (pkg / n).write_text(s)
sys.path.insert(0, str(pathlib.Path.cwd()))
for mod in [m for m in list(sys.modules) if m == "ddqpu" or m.startswith("ddqpu.")]:
    del sys.modules[mod]
import ddqpu
print("ddqpu", ddqpu.__version__)

## 3. Configuración

In [ ]:
from pathlib import Path
from ddqpu import sentinel as S
from ddqpu.backend_utils import get_service, _qubit_info

MODE = "hardware"                 # "hardware" | "sim"
CHANNEL, INSTANCE = "ibm_cloud", "open-instance"
BACKEND = "ibm_fez"
MUST_INCLUDE = [142]              # qubits que quieres vigilar siempre
K = 4                             # candidatos en paralelo (mismo costo)
MAX_QPU_S = 20                    # freno de seguridad
BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT = BASE / "centinela"
BITACORA = S.Bitacora(OUT / "bitacora.jsonl")
SEMBRAR_V4 = True                 # agrega a la bitacora las 4 corridas v4 (23-sep-2026) de q142 como historial
pc = S.ProbeConfig(shots=512, shots_cal=2048, n_candidates=K)

## 4. Backend y candidatos

In [ ]:
if MODE == "sim":
    from qiskit_ibm_runtime.fake_provider import FakeFez
    backend = FakeFez()
else:
    backend = get_service(CHANNEL, INSTANCE).backend(BACKEND)
cands = S.choose_candidates(backend, k=K, min_separation=pc.min_separation, must_include=MUST_INCLUDE)
print("Backend:", backend.name, "| candidatos:", cands)
for q in cands:
    i = _qubit_info(backend, q)
    print(f"  q{q:<4} calibracion publicada: T1={i.t1_us:6.1f} us  T2={i.t2_us:6.1f} us  err_lectura={i.ro_err:.3f}")

if SEMBRAR_V4 and backend.name == "ibm_fez" and not BITACORA.history("ibm_fez", 142):
    v4 = [("2026-09-23T04:00Z", 158.6, -27.04, "estable"), ("2026-09-23T04:02Z", 45.4, -25.32, "degradado"),
          ("2026-09-23T04:14Z", 37.6, -20.81, "degradado"), ("2026-09-23T04:16Z", 153.3, -27.05, "estable")]
    for t, t2, f, e in v4:
        BITACORA.append("ibm_fez", {142: {"q": 142, "T2star_us": t2, "detuning_kHz": f}}, {142: {"estado": e}}, when=t)
    print("Bitacora sembrada con las 4 corridas v4 de q142")

## 5. Sondeo (un job)

In [ ]:
circs, specs = S.build_probe(len(cands), __import__("ddqpu").circuits.timing_from_backend(backend), pc)
cost = S.probe_cost(specs)
print(f"{len(circs)} circuitos | ~{cost['qpu_s']:.1f} s QPU | ~USD {cost['usd']:.2f} a precio Pay-As-You-Go")
manifest, counts = S.run_probe(backend, cands, OUT, pc, max_qpu_seconds=MAX_QPU_S, mode=MODE)
print("Job:", manifest["job_id"])

## 6. Diagnóstico y decisión

In [ ]:
probe = S.analyze_probe(manifest, counts)
verdicts = {}
for q, r in probe.items():
    ref = S.reference_state(BITACORA.history(backend.name, q), _qubit_info(backend, q).t2_us)
    verdicts[q] = S.evaluate(r, ref)
    verdicts[q]["referencia"] = ref
print(S.summarize_probe(backend.name, probe, verdicts, manifest["cost"]))
print()
for q in S.rank_live(probe, verdicts):
    v = verdicts[q]
    g = v["ganancia_dd_estimada"]
    print(f"q{q}: {v['estado']} | {'; '.join(v['motivo'])}" + (f" | DD (eco/T2*) = {g:.2f}x" if g else ""))
BITACORA.append(backend.name, probe, verdicts, when=manifest["created_utc"])
best = S.rank_live(probe, verdicts)[0]
print(f"\nQubit recomendado ahora: q{best}")

## 7. Usar el resultado en tu circuito
- **Corrección de fase**: cada `delay` sin DD acumula 2π·f·τ de fase (27 kHz × 10 µs ≈ 1.7 rad en q142).
  `compensate_detuning` inserta un `rz` virtual después de cada `delay` (costo cero).
- **Qubit**: pasa `best` como `force_target` al notebook v4 o como `initial_layout` a tu transpilación.

In [ ]:
from qiskit import QuantumCircuit
f = probe[best]["detuning_kHz"] or 0.0
demo = QuantumCircuit(1); demo.h(0); demo.delay(2500, 0, unit="dt"); demo.h(0)   # 10 us de espera
fixed = S.compensate_detuning(demo, {0: f}, backend.target.dt)
print(f"detuning q{best} = {f:+.2f} kHz -> fase acumulada en 10 us = {2*3.14159*f*1e3*10e-6:+.3f} rad")
print(fixed.draw(output="text"))

## 8. Historial por qubit

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3.6))
for q in cands:
    h = BITACORA.history(backend.name, q)
    if not h: continue
    ax.plot(range(len(h)), [x["T2star_us"] or 0 for x in h], "-", lw=0.8, label=f"q{q}")
    for i, x in enumerate(h):
        ax.plot(i, x["T2star_us"] or 0, "o", color="#15803d" if x["estado"] == "estable" else "#b91c1c")
ax.set_xlabel("sondeo #"); ax.set_ylabel("T2* (us)"); ax.set_title("Bitacora: verde = estable, rojo = degradado")
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT / "bitacora.png", dpi=130)